# 01. Сбор данных

**ЛР 2, шаг 6.** Ноутбук загружает пространственные данные из OpenStreetMap и сохраняет их в `data/raw/`.

Код адаптирован для проекта **«Аналитика ДТП и опасных участков»**. Мы скачиваем дорожную сеть, светофоры и «лежачие полицейские» для города Уфа.

Данные OpenStreetMap распространяются по лицензии ODbL. На картах и в отчётах указывайте: «© участники OpenStreetMap».

## Настройка среды

Ноутбук рассчитан на **Google Colab** (как получить бесплатный доступ: приложение Б к заданию ЛР 2). Он работает и локально, в Jupyter или VS Code.

Что делает ячейка ниже:
1. В Colab ставит недостающие библиотеки и подключает Google Диск. Colab попросит разрешение на доступ к Диску: выберите тот же аккаунт Google, под которым открыт Colab.
2. Находит рабочую папку группы на Диске. Все данные и результаты сохраняются туда, поэтому после отключения сессии ничего не пропадёт.
3. Создаёт внутри неё ту же структуру папок, что и в репозитории: `data/raw`, `data/manual`, `data/processed`, `docs/img`, `docs/maps`.

**Перед первым запуском** впишите в `PROJECT_FOLDER` имя папки группы на Google Диске.

Запуск всего ноутбука: **Среда выполнения → Выполнить все** (Runtime → Run all).

In [1]:
# === Настройка среды: запустите эту ячейку первой ===
import sys, subprocess
from pathlib import Path

# Имя папки группы в «Моём диске» Google. Обычно совпадает с именем репозитория.
PROJECT_FOLDER = "pm-lab-5-CIA"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", 'geopandas>=1.0', 'osmnx>=2.0', 'mapclassify'], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive") / PROJECT_FOLDER
    if not BASE.exists():
        raise FileNotFoundError(
            f"На Google Диске нет папки «{PROJECT_FOLDER}». Создайте её или добавьте ярлык "
            "общей папки группы в «Мой диск» (приложение Б, шаг 2) и запустите ячейку снова.")
else:
    BASE = Path("..")   # локальный запуск из папки notebooks/ репозитория

RAW, MANUAL, PROCESSED = BASE / "data" / "raw", BASE / "data" / "manual", BASE / "data" / "processed"
IMG, MAPS = BASE / "docs" / "img", BASE / "docs" / "maps"
for folder in (RAW, MANUAL, PROCESSED, IMG, MAPS):
    folder.mkdir(parents=True, exist_ok=True)
print("Среда:", "Google Colab" if IN_COLAB else "локальная", "· рабочая папка:", BASE.resolve())

Mounted at /content/drive
Среда: Google Colab · рабочая папка: /content/drive/MyDrive/pm-lab-5-CIA


In [2]:
from datetime import datetime, timezone
import json

import geopandas as gpd
import pandas as pd
import osmnx as ox

# === Параметры вашего варианта ===
PLACE = "Уфа, Россия"          # территория анализа (так, как её находит Nominatim)
DISTRICT_ADMIN_LEVEL = "9"            # уровень районов города в OSM (в городах России обычно 9)

# Слои для анализа ДТП: дороги, светофоры, лежачие полицейские
LAYERS = {
    "roads":           {"highway": ["motorway", "trunk", "primary", "secondary", "tertiary", "residential"]},
    "traffic_signals": {"highway": "traffic_signals"},
    "speed_bumps":     {"traffic_calming": "bump"},
}
# Атрибуты, которые сохраняем
KEEP_COLUMNS = ["name", "highway", "surface", "lanes", "maxspeed", "traffic_calming"]

ox.settings.use_cache = True
ox.settings.cache_folder = str(RAW / "cache")
ox.settings.requests_timeout = 300
ox.settings.log_console = False

In [3]:
def to_layer(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """Приводит результат osmnx к простой таблице: тип и id объекта OSM, нужные атрибуты, геометрия."""
    gdf = gdf.reset_index()
    gdf = gdf.rename(columns={"element": "osm_type", "id": "osm_id",
                              "element_type": "osm_type", "osmid": "osm_id"})
    cols = ["osm_type", "osm_id"] + [c for c in KEEP_COLUMNS if c in gdf.columns] + ["geometry"]
    gdf = gdf[cols].copy()
    for c in gdf.columns:
        if c != "geometry" and gdf[c].dtype == object:
            gdf[c] = gdf[c].map(lambda v: v if v is None or isinstance(v, str) else str(v))
    return gdf.set_crs(4326, allow_override=True)

log = {"place": PLACE, "downloaded_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
       "source": "OpenStreetMap via Overpass API (osmnx)", "license": "ODbL 1.0", "layers": {}}

## Граница территории и районы

In [4]:
boundary = ox.geocode_to_gdf(PLACE)[["display_name", "geometry"]]
boundary.to_file(RAW / "boundary.gpkg", driver="GPKG")

admin = ox.features_from_place(PLACE, tags={"boundary": "administrative"})
admin = admin[(admin.get("admin_level") == DISTRICT_ADMIN_LEVEL)
              & admin.geometry.geom_type.isin(["Polygon", "MultiPolygon"])]
districts = to_layer(admin)
districts.to_file(RAW / "districts.gpkg", driver="GPKG")

log["layers"]["boundary"] = len(boundary)
log["layers"]["districts"] = len(districts)
print(f"Граница: {len(boundary)}, районов: {len(districts)}")
districts[["name"]].head(20)

Граница: 1, районов: 7


,name
0,Советский район
1,Октябрьский район
2,Дёмский район
3,Ленинский район
4,Кировский район
5,Орджоникидзевский район
6,Калининский район


## Тематические слои

Загрузка крупного города может занять несколько минут. Если сервер Overpass не отвечает (ошибка 429 или 504), подождите и запустите ячейку ещё раз: уже загруженное возьмётся из кэша.

In [5]:
for name, tags in LAYERS.items():
    gdf = to_layer(ox.features_from_place(PLACE, tags=tags))
    gdf.to_file(RAW / f"{name}.gpkg", driver="GPKG")
    log["layers"][name] = len(gdf)
    print(f"{name}: {len(gdf)} объектов, типы геометрий: {gdf.geom_type.value_counts().to_dict()}")

(RAW / "_download_log.json").write_text(json.dumps(log, ensure_ascii=False, indent=2), encoding="utf-8")
log

roads: 4383 объектов, типы геометрий: {'LineString': 4383}
traffic_signals: 439 объектов, типы геометрий: {'Point': 439}
speed_bumps: 59 объектов, типы геометрий: {'Point': 59}


{'place': 'Уфа, Россия',
 'downloaded_at': '2026-11-10T08:43:57+00:00',
 'source': 'OpenStreetMap via Overpass API (osmnx)',
 'license': 'ODbL 1.0',
 'layers': {'boundary': 1,
  'districts': 7,
  'roads': 4383,
  'traffic_signals': 439,
  'speed_bumps': 59}}

## Быстрая проверка на карте

In [6]:
roads = gpd.read_file(RAW / "roads.gpkg")
m = boundary.explore(color="gray", style_kwds={"fillOpacity": 0.05})
roads.explore(m=m, color="red", tooltip=["name", "highway"])
m

## Если Overpass недоступен

1. Откройте [Overpass Turbo](https://overpass-turbo.eu), выполните запрос по своей территории и сохраните результат: **Export → GeoJSON**. Положите файл в папку `data/raw` (в Colab — на Google Диске: значок папки слева → `drive/MyDrive/<папка группы>/data/raw`) и прочитайте его через `gpd.read_file(RAW / "имя_файла.geojson")`.
2. Либо скачайте выгрузку региона с [Geofabrik](https://download.geofabrik.de) и откройте её в QGIS (слои points, multipolygons), отфильтруйте по тегам и сохраните в GeoPackage.

Каким бы способом вы ни получили данные, запишите его в `data/README.md`, чтобы другой участник группы мог повторить загрузку.

## Что записать в реестр источников

Для каждого слоя внесите или обновите карточку в `data/sources.md`: дата загрузки (из `_download_log.json`), число объектов, лицензия, система координат.